In [0]:
%run ../00-common/config

In [0]:
from pyspark.sql import functions as F

items = spark.table(f"{catalog_name}.{silver_schema}.order_items")
orders = spark.table(f"{catalog_name}.{silver_schema}.orders").select("order_id", "order_purchase_timestamp")

fact_order_items = (items
    # trashego purchase_date nga orders (merr daten e orders permes order_id qe dhe artikujt te lidhen me dim_date)
    .join(orders, on="order_id", how="left")
    .withColumn("date_key", F.date_format("order_purchase_timestamp", "yyyyMMdd").cast("int"))
    .select(
        "order_id", "order_item_id", "product_id", "seller_id",
        "date_key", "price", "freight_value"
    ))

(fact_order_items.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{catalog_name}.{gold_schema}.fact_order_items"))
print(f"Wrote {fact_order_items.count():,} items")